# Fast convolution

So, the english is not very good, I will improve that

In [1]:
import itertools

import sympy as sy
import numpy as np

In [2]:
import fitz
from IPython.display import YouTubeVideo

In [3]:
from utils import plot_pdf

In [4]:
doc = fitz.open('Blahut_2010_Fast algorithms for signal processing.pdf')

The base for this tutorial is the book "Fast Algorithms or Signal Processing" of Blahut.
Is an excellent book but full of examples but, of course, do not explain everything in details.
For that parts I will quote another books and videos from YouTube.

The book puts first all theory and only after that we have a detailed example. To better understanding in this tutorial we have quotes from theory and examples together fallowing the sequence of commands

This example will not work for different vector sizes.

## Start

Size of vectors

In [39]:
d_num = 3
g_num = 3

In [40]:
b_degree = d_num + g_num - 1
b_degree

5

In [46]:
# bi = [1, 2, -2, sy.Rational(1, 2)]
bi = [0, 1, -1, -2, sy.oo]
sy.Matrix(bi)

Matrix([
[ 0],
[ 1],
[-1],
[-2],
[oo]])

Example of vectors for the convolution

In [47]:
d_values = list(range(1, d_num+1))
g_values = list(range(1, g_num+1))
print(d_values, g_values)

[1, 2, 3] [1, 2, 3]


Polynomial degree

In [48]:
d_degree = d_num - 1
g_degree = g_num - 1
print(d_degree, g_degree)

2 2


In [49]:
di = sy.Matrix(sy.symbols(" ".join(f"d_{i}"for i in range(d_num))))
di

Matrix([
[d_0],
[d_1],
[d_2]])

In [50]:
gi = sy.Matrix(sy.symbols(" ".join(f"g_{i}"for i in range(g_num))))
gi

Matrix([
[g_0],
[g_1],
[g_2]])

In [60]:
_a_mtx = [[(b **e) for e, d in enumerate(di)] for b in bi if b != sy.oo]
_a_inf = [[0] * (len(di) - 1) + [1]]
a_mtx = sy.Matrix(_a_mtx + _a_inf)
a_mtx

Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[1, -2, 4],
[0,  0, 1]])

In [61]:
_b_mtx = [[(b **e) for e, d in enumerate(gi)] for b in bi if b != sy.oo]
_b_inf = [[0] * (len(gi) - 1) + [1]]
b_mtx = sy.Matrix(_b_mtx + _b_inf)
b_mtx

Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[1, -2, 4],
[0,  0, 1]])

In [62]:
_c_mtx = [[(b **e) for e in range(b_degree)] for b in bi if b != sy.oo]
_c_inf = [[0] * (b_degree - 1) + [1]]
c_mtx = sy.Matrix(_c_mtx + _c_inf)
c_mtx

Matrix([
[1,  0, 0,  0,  0],
[1,  1, 1,  1,  1],
[1, -1, 1, -1,  1],
[1, -2, 4, -8, 16],
[0,  0, 0,  0,  1]])

In [64]:
c_inv0 = c_mtx.inv()
c_inv0

Matrix([
[   1,   0,   0,    0,  0],
[ 1/2, 1/3,  -1,  1/6, -2],
[  -1, 1/2, 1/2,    0, -1],
[-1/2, 1/6, 1/2, -1/6,  2],
[   0,   0,   0,    0,  1]])

In [67]:
s = sy.MatMul(c_mtx, bg_mtx, a_mtx, sy.Matrix(di))
s

ShapeError: ('Matrices are not aligned', 3, 5)

## Example

In [31]:
subs = {k[0]: v for k, v in zip(di.tolist()+gi.tolist(), d_values + g_values)}
subs

{d_0: 1, d_1: 2, d_2: 3, g_0: 1, g_1: 2}

In [32]:
si = s.subs(subs)
si

Matrix([
[ 2,  0,  0,  0],
[-1, -2,  2, -1],
[-2, -1, -3,  0],
[ 1,  1,  1,  1]])*Matrix([
[1/2,    0,   0,   0],
[  0, -3/2,   0,   0],
[  0,    0, 1/6,   0],
[  0,    0,   0, 5/6]])*Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[1,  2, 4]])*Matrix([
[1],
[2],
[3]])

In [33]:
sy.expand(sx)

d_0*g_0 + d_0*g_1*x + d_1*g_0*x + d_1*g_1*x**2 + d_2*g_0*x**2 + d_2*g_1*x**3

Let's compare the output polynomial matrix from direct and winograd method

In [34]:
sy.Matrix(np.convolve(np.array(di).reshape(-1), np.array(gi).reshape(-1)))

Matrix([
[          d_0*g_0],
[d_0*g_1 + d_1*g_0],
[d_1*g_1 + d_2*g_0],
[          d_2*g_1]])

In [35]:
se = sy.MatMul(c_mtx, bg_mtx, a_mtx, sy.Matrix(di), evaluate=True)
se

Matrix([
[          d_0*g_0],
[d_0*g_1 + d_1*g_0],
[d_1*g_1 + d_2*g_0],
[          d_2*g_1]])

Comparing numerical outputs from direct and winograd method

In [36]:
sy.Matrix(np.convolve(d_values, g_values))

Matrix([
[1],
[4],
[7],
[6]])

In [37]:
se.subs(subs)

Matrix([
[1],
[4],
[7],
[6]])